# Skinwise — Train the Acne Detector (YOLOv11-nano)

Run this in **Google Colab** with a **GPU** runtime: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

At the end it downloads **`acne_yolo11n_v1.onnx`** — send that file back and it gets wired straight into the engine (drop it at `skin-cv-service/models/`).

Deterministic by design: fixed seed, single class `lesion`, exported to ONNX opset 17 (NMS handled deterministically inside the service, not in the graph).

In [ ]:
!pip -q install ultralytics==8.3.0 onnx onnxslim onnxruntime roboflow

## 1. Get an acne dataset (YOLO format)

Easiest path — **Roboflow Universe**: https://universe.roboflow.com/search?q=acne%20detection

1. Make a free account.
2. Open an **object-detection** acne dataset (bounding boxes around lesions).
3. Click **Download → YOLOv11 → show download code** and paste YOUR snippet below (replace the placeholder one).

Or, if you already have a YOLO-format dataset, upload it and set `DATA_YAML` to its `data.yaml`.

In [ ]:
from roboflow import Roboflow
# >>> PASTE YOUR ROBOFLOW SNIPPET HERE (from Download code). Example shape: <<<
rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
project = rf.workspace("WORKSPACE").project("acne-PROJECT")
dataset = project.version(1).download("yolov11")
DATA_YAML = dataset.location + "/data.yaml"
print("dataset:", DATA_YAML)

In [ ]:
import random, numpy as np, torch
random.seed(0); np.random.seed(0); torch.manual_seed(0)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data=DATA_YAML, imgsz=640, epochs=60, batch=16,
    seed=0, deterministic=True, single_cls=True,
    project="acne", name="run", exist_ok=True,
)

In [ ]:
# Export to ONNX (opset 17, NMS excluded — the service does deterministic NMS).
import shutil
best = "acne/run/weights/best.pt"
YOLO(best).export(format="onnx", opset=17, imgsz=640, nms=False, simplify=True)
shutil.copy("acne/run/weights/best.onnx", "acne_yolo11n_v1.onnx")

import hashlib
print("sha256:", hashlib.sha256(open("acne_yolo11n_v1.onnx", "rb").read()).hexdigest())
from google.colab import files
files.download("acne_yolo11n_v1.onnx")

## Done
Send me `acne_yolo11n_v1.onnx` (and the sha256 printed above). I'll add it to the engine, wire acne into the 8th concern, and redeploy.

**Note on quality:** a public dataset gives a *demo/Beta* detector — decent on clear cases, weaker on darker skin tones. Production accuracy comes from labelling your own customer photos over time (over-sampling Fitzpatrick IV–VI).